# GOAL: Build SARIMAX model

Note: using "previous day" would NOT mean that today is used to predict tomorrow. It means the prediction for today is used to predict tomorrow.

In [31]:
#import statements

#import everything!
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime, timedelta
from itertools import product
from copy import deepcopy
from PreRun import PreRun, PostRun
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
import random


#path to data
read_path = "../../../../data_ds_project/parquet_cleaned_energy"
#systems
good_systems_list = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 1332, 4902, 4903]
reader_types = ["meter", "inverter", None]
#systems_cleaned
systems_cleaned = pd.read_csv("../../../data/core/systems_cleaned.csv")


In [28]:
system_id = 4
reader_type = None

prerun = PreRun(system_id = system_id, meter_or_inverter = reader_type, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
prerun.fill_missing_hours()
prerun.add_energy_features_only(include_hour_cyclic=True)
prerun.add_weather_features_only()

all_data = prerun.amended_data
end_days = prerun.good_end_days_naive(7)
#print(end_days)

big_train_dates, big_test_dates = end_days.iloc[:int(len(end_days)*0.8)]['date'], end_days.iloc[int(len(end_days)*0.8):]['date']
big_train_dates = pd.to_datetime(big_train_dates)
big_test_dates = pd.to_datetime(big_test_dates)
big_train_dates_set = set(big_train_dates)
big_test_dates_set = set(big_test_dates)



In [ ]:
#edit this part!!!  
error_list = []
random.seed(42)
pred_date_set = set(random.sample(list(big_test_dates_set),10))
for pred_date in pred_date_set:
    train_data = all_data.loc[(all_data['time'] < pred_date - timedelta(days=1)) &
                            (all_data['time'] >= pred_date - timedelta(days=7))].reset_index(drop=True)
    ho_data = all_data[(all_data['time'] >= pred_date - timedelta(days=1)) 
                        & (all_data['time'] <= pred_date)].reset_index(drop=True)

    y_pred, error = use_sarimax(train_data, ho_data, p=1, d=0, q=1)
    error_list.append(error)

c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmo

In [30]:
print(error_list)
print(np.mean(error_list))

[np.float64(0.006035626863027035), np.float64(0.009995840407225252), np.float64(0.01736349310660729), np.float64(0.022425061155002216), np.float64(0.008772556137935414), np.float64(0.015980508339009054), np.float64(0.020707370000753064), np.float64(0.01072087851079156), np.float64(0.015926424303676887), np.float64(0.005367855891696678)]
0.013329561471572443


In [4]:
def use_sarimax(df_train, df_ho, p=0, d=0, q=0):

    #separate out the exogenous variables
    df_train = df_train.set_index('time')
    df_ho = df_ho.set_index('time')
    energy_train = df_train['energy']
    energy_ho = df_ho['energy']

    exog_train = df_train.drop(columns=['energy'])
    exog_ho = df_ho.drop(columns=['energy'])

    #fit the model
    model_sarimax = SARIMAX(energy_train, exog=exog_train, order=(p,d,q)).fit()
    #predict
    y_pred = model_sarimax.forecast(len(df_ho), exog=exog_ho) #should be only 2 steps. hopefully no na's

    error = PostRun.custom_error(energy_ho, y_pred, 1,2)

    return y_pred, error


    